# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:\n", metadata.description)
print("Version:", getattr(metadata, 'version', 'N/A'))
print("Date Published:", getattr(metadata, 'datePublished', 'N/A'))

## 2. Data Overview
Review available record sets and fields, identified by their `@id`.

Each Croissant schema describes one or more `RecordSet` entries, each exposing fields (columns) and their metadata. We'll enumerate all record sets, list their `@id`s, and show field details.

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets (by @id):")
record_sets_info = []
for record_set in dataset.record_sets:
    print(f"  - RecordSet Name: {record_set.name}, @id: {record_set.id}")
    field_ids = []
    for field in record_set.fields:
        print(f"    * Field: {field.name}, @id: {field.id}, DataType: {getattr(field, 'data_type', 'unknown')}")
        field_ids.append(field.id)
    record_sets_info.append({'record_set_id': record_set.id, 'record_set_name': record_set.name, 'field_ids': field_ids})
# Save for later references
all_record_set_ids = [rs['record_set_id'] for rs in record_sets_info]

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. All record sets and field references use their `@id` values as required.

Below, we load all available record sets found in the previous step.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet @id: {record_set_id}")
    if len(dataframes[record_set_id].columns) > 0:
        print(f"  Columns (@id): {list(dataframes[record_set_id].columns)}")
    else:
        print("  [No columns - possibly empty RecordSet]")
# Pick one record set as default for EDA (if more than one, choose the first with data)
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty and df.shape[1] > 0:
        main_record_set_id = rid
        break
if main_record_set_id:
    print(f"\nExample rows from RecordSet {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record sets with data were found.")

## 4. Exploratory Data Analysis (EDA)
We select numeric fields by their `@id` and perform filtering, normalization, and simple groupby analysis.

Fill in the `numeric_field_id` and `group_field_id` with those observed above. Adjust the threshold depending on the data distribution.

In [ ]:
# Choose a numeric field @id for analysis from this record set
df = dataframes[main_record_set_id]

# Attempt to automatically pick a numeric field by inspecting dtypes
numeric_field_id = None
for col in df.columns:
    # Check if column can be converted to numeric (float/int) for at least 80% of entries
    try:
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notnull().sum() / len(converted) > 0.8:
            numeric_field_id = col
            break
    except Exception:
        continue

if numeric_field_id is None:
    print("No numeric field found for analysis in this record set.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")
    # Convert to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum() > 0 else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (N={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalization
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized column for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Pick a group-able field (string or categorical)
    group_field_id = None
    for col in df.columns:
        if col == numeric_field_id:
            continue
        if df[col].dtype == 'object' and df[col].nunique() > 1 and df[col].nunique() < (len(df) // 2):
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std']).reset_index().sort_values('count', ascending=False)
        print(f"Aggregated statistics for {numeric_field_id} by group {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouped analysis.")

## 5. Visualization
Visualize the distribution of numeric values and relationships to group fields, where applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 6))
        # Only plot top 8 categories
        top_groups = df[group_field_id].value_counts().index[:8]
        plot_df = df[df[group_field_id].isin(top_groups)]
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=plot_df)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field_id} by {group_field_id} (top 8 groups)')
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using the Croissant schema with the `mlcroissant` library. We identified available record sets and fields by their `@id`, loaded data into pandas DataFrames, performed preliminary data cleaning and exploratory data analysis, and visualized key distributions.

- Always reference record sets, fields, and columns by their `@id` for reproducibility.
- Consult the Croissant schema and dataset documentation for more advanced analysis.

Further exploration might include advanced statistical modeling, feature engineering, or geospatial analyses tailored to the dataset's structure and research questions related to rangeland management, gender inclusion, and knowledge adoption in Northern Kenya.